# Traffic Sign Detection

## Step 1: Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — go to Runtime > Change runtime type > T4 GPU')

Wed Jun 17 03:20:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Mount Google Drive

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

Mounted at /content/drive
Drive mounted at /content/drive


## Step 3: Install Dependencies

In [ ]:
# ── Cell 3: Install dependencies ───────────────────────────────────
!pip install ultralytics opencv-python-headless -q
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics version: 8.4.69


Ultralytics 8.4.69 is the version used throughout this project. This version
includes the DFL (Distribution Focal Loss) regression head which improves bounding box
precision compared to earlier YOLO versions, and the C2f backbone blocks that give
richer gradient flow than C3 blocks used in YOLOv5.

## Step 4: Copy Dataset to Local Storage

In [ ]:
import shutil
from pathlib import Path

DRIVE_DATASET = '/content/drive/MyDrive/Traffic_Signs_Detection_ComputerVision'
LOCAL_DATASET = '/content/dataset'

if not Path(DRIVE_DATASET).exists():
    print(f'ERROR: Folder not found at {DRIVE_DATASET}')
    print('Make sure the folder name in your Drive is exactly: Traffic_Signs_Detection_ComputerVision')
else:
    if Path(LOCAL_DATASET).exists():
        shutil.rmtree(LOCAL_DATASET)
    print('Copying dataset to Colab local storage (faster for training)...')
    shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
    print('Done.')

# Verify
train_imgs = list(Path(LOCAL_DATASET + '/car/train/images').glob('*.jpg'))
print(f'Train images found: {len(train_imgs)}')
print(f'Video found: {Path(LOCAL_DATASET + "/video.mp4").exists()}')

Copying dataset to Colab local storage (faster for training)...
Done.
Train images found: 3530
Video found: True


3,530 training images confirmed. The full split is:
- Train: 3,530 images
- Validation: 801 images
- Test: 638 images (held out, not used until evaluation)

The dashcam video is also confirmed present.

## Step 5: Patch data.yaml with Absolute Paths

In [ ]:
import yaml
from pathlib import Path

DATA_YAML    = Path('/content/dataset/car/data.yaml')
PATCHED_YAML = Path('/content/data_patched.yaml')

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

car_dir = DATA_YAML.parent
cfg['train'] = str(car_dir / 'train' / 'images')
cfg['val']   = str(car_dir / 'valid' / 'images')
cfg['test']  = str(car_dir / 'test'  / 'images')

with open(PATCHED_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Classes:', cfg['names'])
print('Train path:', cfg['train'])
print('Patched yaml ready.')

Classes: ['Green Light', 'Red Light', 'Speed Limit 10', 'Speed Limit 100', 'Speed Limit 110', 'Speed Limit 120', 'Speed Limit 20', 'Speed Limit 30', 'Speed Limit 40', 'Speed Limit 50', 'Speed Limit 60', 'Speed Limit 70', 'Speed Limit 80', 'Speed Limit 90', 'Stop']
Train path: /content/dataset/car/train/images
Patched yaml ready.


All 15 classes loaded correctly. The class ordering here defines the numeric class IDs
used in all YOLO label files: 0 = Green Light, 1 = Red Light, 2 = Speed Limit 10, and

Worth noting: Speed Limit 10 is class ID 2. Despite being present in the dataset, it
only has 19 training samples compared to 585 for Red Light. That is a 30x imbalance
and it directly causes Speed Limit 10 to have the lowest detection accuracy of all classes.

## Step 6: Train YOLOv8n (around 30 to 40 minutes on T4)

Fine-tunes a YOLOv8 nano model starting from COCO pretrained weights. Key parameter decisions:

| Parameter | Value | Reason |
|-----------|-------|--------|
| `epochs` | 50 | Upper limit; early stopping handles the actual cutoff |
| `patience` | 10 | Stops training if val mAP does not improve for 10 consecutive epochs |
| `imgsz` | 416 | Matches the dataset native resolution |
| `batch` | 16 | Fits T4 VRAM comfortably |
| `cls` | 1.5 | Default is 0.5. Increased to penalise misclassification harder. Speed limit signs look visually similar, so getting the class right matters more than a slightly imprecise box |
| `mosaic` | 1.0 | Pastes 4 images together per training tile, increasing scale and context variety |
| `mixup` | 0.1 | Blends two images with soft labels to improve generalisation |
| `degrees` | 10.0 | Small rotation to simulate camera angle variation |

The best checkpoint (by validation mAP@50) is saved automatically to `weights/best.pt`.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data=str(PATCHED_YAML),
    epochs=50,
    imgsz=416,
    batch=16,
    project='/content/outputs',
    name='yolov8n_traffic',
    patience=10,
    augment=True,
    cls=1.5,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    exist_ok=True,
    device=0,
)

print('Training complete!')
print('Best weights exist:', Path('/content/outputs/yolov8n_traffic/weights/best.pt').exists())

Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_patched.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_traffic, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

**What to look at in the training output above:**

The table printed each epoch has these key columns:
- `box_loss`: how well the model places bounding boxes. Should decrease each epoch.
- `cls_loss`: how well the model identifies the sign class. Also decreases, but more slowly
  since classification is harder than localisation.
- `mAP50`: validation mAP at IoU 0.5. It should rise steadily
  and then flatten out when early stopping triggers.

If training stopped before epoch 50, early stopping kicked in because val mAP did not
improve for 10 consecutive epochs. The best checkpoint is saved when val mAP peaks, not at the final epoch.

## Step 7: Evaluate on Test Set and Export Metrics

Loads the best checkpoint and runs it on 638 test images that the model never saw during
training or validation.

Per-class precision, recall, mAP@50, and mAP@50-95 are saved to `metrics.json` for
the Streamlit dashboard.

In [ ]:
import json
from ultralytics import YOLO

model = YOLO('/content/outputs/yolov8n_traffic/weights/best.pt')

print('Running evaluation on test set...')
metrics = model.val(data=str(PATCHED_YAML), split='test', verbose=False)
box = metrics.box

class_names = cfg['names']

cm_data = None
try:
    cm = metrics.confusion_matrix.matrix
    if cm is not None:
        cm_data = cm.tolist()
except:
    pass

output = {
    'class_names': class_names,
    'precision':   box.p.tolist(),
    'recall':      box.r.tolist(),
    'ap50':        box.ap50.tolist(),
    'ap':          box.ap.tolist(),
    'map50':       float(box.map50),
    'map':         float(box.map),
    'confusion_matrix': cm_data,
    'pr_curves': {},
}

with open('/content/outputs/metrics.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f'mAP@50    = {box.map50:.4f}')
print(f'mAP@50-95 = {box.map:.4f}')
print('Saved: /content/outputs/metrics.json')

Running evaluation on test set...
Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 569.2±179.9 MB/s, size: 20.8 KB)
val: Scanning /content/dataset/car/test/labels... 638 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 638/638 1.2Kit/s 0.5s
val: New cache created: /content/dataset/car/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 8.0it/s 5.0s
                   all        638        770      0.864      0.886      0.932      0.767
Speed: 0.7ms preprocess, 2.3ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detect/val
mAP@50    = 0.9316
mAP@50-95 = 0.7670
Saved: /content/outputs/metrics.json


**Result: mAP@50 = 0.9316, mAP@50-95 = 0.7670**

What these numbers mean:

- **mAP@50 = 0.932**: Across all 15 classes, the model correctly detects and identifies
  traffic signs with at least 50% bounding box overlap 93.2% of the time.

- **mAP@50-95 = 0.767**: The stricter metric that averages mAP across IoU thresholds from
  0.50 to 0.95. This measures how precisely the boxes are drawn, not just whether the
  detection is roughly correct. The gap between 0.932 and 0.767 is normal and shows the
  model is accurate at detection but the box edges are not always pixel-perfect.

- **Inference speed: 2.3ms per image**: That is roughly 430 frames per second. A dashcam
  runs at 30 FPS, so this model has about 14x more compute headroom than it needs. Even
  on less powerful hardware, real-time deployment is feasible.

- **Precision = 0.864, Recall = 0.886**: Recall is slightly higher than precision.
  The model misses fewer signs than it falsely triggers on, which is the right trade-off
  for a safety application.

## Step 8: Export Confidence Score Distributions

Runs the trained model on 400 test images with a low confidence threshold of 0.1.
The low threshold captures borderline detections as well, not just confident ones.
This gives a full distribution of confidence scores per class.

The Streamlit dashboard uses this CSV to draw violin plots showing how certain the model
is about each sign type. Wide, high-confidence violins mean the model is reliable for
that class. Sparse or low-score violins signal uncertainty, usually from limited training
data or visual similarity with other classes.

In [ ]:
import csv
from pathlib import Path

test_imgs = list(Path('/content/dataset/car/test/images').glob('*.jpg'))[:400]
records = []

for img_path in test_imgs:
    results = model(str(img_path), conf=0.1, verbose=False)
    for r in results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            cid  = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            name = class_names[cid] if cid < len(class_names) else str(cid)
            records.append({'class_id': cid, 'class_name': name, 'confidence': conf})

with open('/content/outputs/confidence_detections.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['class_id', 'class_name', 'confidence'])
    writer.writeheader()
    writer.writerows(records)

print(f'Saved {len(records)} detections to confidence_detections.csv')

Saved 563 detections to confidence_detections.csv


563 detections collected across 400 test images. That is about 1.4 detections per
image on average, which makes sense since most dashcam frames contain one or two signs
at most.

The low threshold of 0.1 means borderline detections are included. These low-confidence
cases are what make the violin plots in the dashboard informative. A class with all its
detections bunched above 0.85 is reliable. A class with detections spread from 0.1 to
0.9 is one the model is uncertain about.

## Step 9: Run Inference on the Dashcam Video

Processes every frame of the 508-frame dashcam video (30 FPS, around 17 seconds).
Detected signs get coloured bounding boxes and class labels drawn on each frame.
The annotated output is saved as `processed_video.mp4`.

Per-frame detection counts and per-class totals are saved to `video_stats.json`
for the dashboard timeline chart.

In [ ]:
import cv2, json

CLASS_COLORS = [
    '#2ecc71','#e74c3c','#3498db','#9b59b6','#f39c12',
    '#1abc9c','#e67e22','#d35400','#c0392b','#16a085',
    '#8e44ad','#2980b9','#27ae60','#f1c40f','#7f8c8d',
]

def hex_to_bgr(h):
    h = h.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return (b, g, r)

VIDEO_IN  = '/content/dataset/video.mp4'
VIDEO_OUT = '/content/outputs/processed_video.mp4'

cap   = cv2.VideoCapture(VIDEO_IN)
fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out = cv2.VideoWriter(VIDEO_OUT, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

dpf, class_counts, frame_idx = [], {}, 0
print(f'Processing {total} frames at {fps:.1f} FPS...')

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = model(frame_rgb, conf=0.25, verbose=False)
    dets = []
    for r in results:
        if r.boxes is None: continue
        for box in r.boxes:
            cid  = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            x1,y1,x2,y2 = [int(v) for v in box.xyxy[0].tolist()]
            name = class_names[cid] if cid < len(class_names) else str(cid)
            dets.append((cid, conf, x1, y1, x2, y2, name))
            class_counts[name] = class_counts.get(name, 0) + 1

    dpf.append(len(dets))
    bgr = frame.copy()
    for cid, conf, x1, y1, x2, y2, name in dets:
        color = hex_to_bgr(CLASS_COLORS[cid % len(CLASS_COLORS)])
        cv2.rectangle(bgr, (x1, y1), (x2, y2), color, 2)
        label = f'{name} {conf:.2f}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(bgr, (x1, y1-th-6), (x1+tw+4, y1), color, -1)
        cv2.putText(bgr, label, (x1+2, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
    out.write(bgr)
    frame_idx += 1
    if frame_idx % 50 == 0:
        print(f'  Frame {frame_idx}/{total}')

cap.release()
out.release()

stats = {'total_frames': total, 'fps': fps, 'detections_per_frame': dpf, 'class_counts': class_counts}
with open('/content/outputs/video_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f'Done. Detections per class: {class_counts}')

Processing 508 frames at 30.0 FPS...
  Frame 50/508
  Frame 100/508
  Frame 150/508
  Frame 200/508
  Frame 250/508
  Frame 300/508
  Frame 350/508
  Frame 400/508
  Frame 450/508
  Frame 500/508
Done. Detections per class: {'Speed Limit 50': 65, 'Green Light': 238, 'Speed Limit 20': 54, 'Stop': 3, 'Speed Limit 10': 8, 'Speed Limit 30': 4, 'Speed Limit 70': 1, 'Speed Limit 40': 5, 'Speed Limit 90': 30, 'Speed Limit 80': 13, 'Red Light': 156, 'Speed Limit 120': 31, 'Speed Limit 110': 2, 'Speed Limit 60': 8}


**508 frames processed at 30 FPS.**

Detection breakdown:

| Class | Count | What this tells us |
|-------|-------|--------------------|
| Green Light | 238 | The road passes through multiple signalised intersections |
| Red Light | 156 | About 1 in 3 green light frames also has a red light nearby |
| Speed Limit 50 | 65 | Most of the footage is in a 50 km/h zone |
| Speed Limit 90 | 30 | A short highway section in the video |
| Speed Limit 20 | 54 | School zone or residential street section |
| Speed Limit 10 | 8 | Despite having only 19 training samples, the model still detected this class 8 times in real footage. It learned something, just not enough to be reliable. |

The `detections_per_frame` list in `video_stats.json` is what the dashboard uses to
draw the timeline chart showing detection activity across the video.

## Step 10: Package and Save Outputs to Google Drive

In [ ]:
import shutil
from pathlib import Path

print('Zipping outputs...')
shutil.make_archive('/content/outputs_for_download', 'zip', '/content/outputs')

DRIVE_OUT = '/content/drive/MyDrive/BENG_RMI_outputs.zip'
shutil.copy('/content/outputs_for_download.zip', DRIVE_OUT)
print(f'Saved to Google Drive: {DRIVE_OUT}')
print()
print('Files inside the zip:')
for p in sorted(Path('/content/outputs').rglob('*')):
    if p.is_file():
        mb = p.stat().st_size / 1024 / 1024
        print(f'  {str(p.relative_to("/content/outputs")):50s}  {mb:.1f} MB')

Zipping outputs...
Saved to Google Drive: /content/drive/MyDrive/BENG_RMI_outputs.zip

Files inside the zip:
  confidence_detections.csv                           0.0 MB
  metrics.json                                        0.0 MB
  processed_video.mp4                                 3.9 MB
  video_stats.json                                    0.0 MB
  yolov8n_traffic/BoxF1_curve.png                     0.4 MB
  yolov8n_traffic/BoxPR_curve.png                     0.2 MB
  yolov8n_traffic/BoxP_curve.png                      0.3 MB
  yolov8n_traffic/BoxR_curve.png                      0.3 MB
  yolov8n_traffic/args.yaml                           0.0 MB
  yolov8n_traffic/confusion_matrix.png                0.3 MB
  yolov8n_traffic/confusion_matrix_normalized.png     0.3 MB
  yolov8n_traffic/labels.jpg                          0.2 MB
  yolov8n_traffic/results.csv                         0.0 MB
  yolov8n_traffic/results.png                         0.3 MB
  yolov8n_traffic/train_batch0.jpg   

## Step 11: Download Zip to Computer

In [ ]:
from google.colab import files

print("Downloading zip file to your local computer...")
files.download('/content/outputs_for_download.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>